# 01 · Synthetic upay world: generation and validation

The generator (`src/datagen/`) simulates 60 days of an upay-like MFS: normal life (salaries on days 1–5,
Friday remittance-style transfers, Eid salami, bills mid-month, cash-in/out at agents), legit look-alikes
(new wallets, shared family phones, f-commerce sellers, family hub wallets, phone changes) and eight injected
fraud scenarios **S1–S8** with ground truth. A ledger enforces balances and KYC limits on every row.

In [1]:
# --- setup: find the repo root (works locally, in Colab and on Kaggle) -------------------------
import os, sys, shutil, subprocess
from pathlib import Path

REPO_URL = ""          # optional: your GitHub repo URL, cloned if the code is not found
ROOT = None
for cand in [Path.cwd(), Path.cwd().parent, *Path("/kaggle/input").glob("*")] if Path("/kaggle/input").exists() else [Path.cwd(), Path.cwd().parent]:
    if (cand / "src" / "pipeline.py").exists():
        ROOT = cand
        break
if ROOT is None and REPO_URL:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "prohori"], check=True)
    ROOT = Path("prohori").resolve()
assert ROOT is not None, "Put this notebook next to the repo (or add the repo as a Kaggle dataset / set REPO_URL)"
if str(ROOT).startswith("/kaggle/input"):          # Kaggle inputs are read-only: work on a copy
    dst = Path("/kaggle/working/prohori")
    shutil.copytree(ROOT, dst, dirs_exist_ok=True)
    ROOT = dst
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

repo root: H:\Tacos Projects\Hackathon\Asking for a Friend


In [2]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40, "display.width", 200)
from src.common.config import load_config, resolve
from src.models import plots
plots.style()
cfg = load_config()                 # config.yaml (seed 42, full scale)
REPORTS = resolve(cfg, "reports_dir")

In [3]:
REBUILD = False      # True = regenerate (seed 42, ~10 s) even if data exists
from src.datagen.generate import build, write, load
data_dir = resolve(cfg, "data_dir")
if REBUILD or not (data_dir / "transactions.parquet").exists():
    write(build(cfg), cfg)
data = load(cfg)
tx, lab, cust = data["transactions"], data["labels"], data["customers"]
print(json.dumps(data["report"], indent=1, default=str)[:2500])

{
 "seed": 42,
 "scale": 1.0,
 "n_transactions": 683148,
 "n_customers": 11573,
 "n_fraud_wallets": 763,
 "fraud_rows": 3664,
 "fraud_share": 0.00536,
 "fraud_rows_by_scenario": {
  "S1": 944,
  "S2": 893,
  "S5": 487,
  "S7": 439,
  "S3": 382,
  "S6": 308,
  "S8": 211
 },
 "cases_by_scenario": {
  "S1": 191,
  "S7": 166,
  "S3": 61,
  "S6": 56,
  "S2": 41,
  "S5": 41,
  "S8": 37
 },
 "status": {
  "SUCCESS": 654030,
  "FAILED": 29118
 },
 "txn_type_mix": {
  "SEND_MONEY": 237256,
  "MOBILE_RECHARGE": 143155,
  "CASH_IN": 90172,
  "PAYMENT": 86410,
  "CASH_OUT": 84986,
  "BILL_PAY": 18226,
  "ADD_MONEY": 15298,
  "SALARY": 5332,
  "REMITTANCE_IN": 2274,
  "FLOAT_TOPUP": 39
 },
 "n_account_events": 3229,
 "n_complaints": 824,
 "seconds_total": 9.9
}


## Validation T1–T14 (`python -m src.validation.run_checks`)

In [4]:
from src.validation.checks import Checker
res = Checker(data, cfg).run(skip_repro=True)      # T14 (re-generates twice) runs in the CLI / pytest
pd.DataFrame([{"test": r["id"], "check": r["name"], "passed": r["passed"]} for r in res])

,test,check,passed
0,T1,schema & keys,True
1,T2,referential integrity,True
2,T3,balance conservation,True
3,T4,KYC limits,True
4,T5,temporal realism,True
5,T6,volume & mix,True
6,T7,amount realism,True
7,T8,fraud prevalence & spread,True
8,T9,label consistency,True
9,T10,scenario signatures,True


In [5]:
for r in res:
    if r["id"] in ("T5", "T10", "T11", "T13"):
        print(r["id"], json.dumps(r["details"], default=str)[:700], "\n")

T5 {"night_share_00_06": 0.053, "peak_hour": 20, "eid_p2p_vs_median_day": 1.96, "salary_on_days_1_10": true, "friday_p2p_vs_weekday": 1.148, "normal_cash_night_share": 0.0005} 

T10 {"S2_new_collector_age_days_max": 2.91, "S2_min_unique_victims": 8, "S3_median_hop_delay_min": 13.3, "S6_swap_before_drain_share": 1.0, "S1_remote_access_share": 0.102, "S1_drain_on_new_device_share": 1.0, "S7_send_on_own_device_share": 0.963, "S8_starts_with_add_money_share": 1.0, "S4_episode_volume_multiplier_median": 10.32, "S4_required_multiplier": 4.0} 

T11 {"aged_mule_share": 0.25, "fraud_night_share": 0.116, "normal_night_share": 0.047, "legit_new_wallets": 765, "legit_device_changes": 2042, "legit_sim_swaps": 91, "benign_fanin_wallet_days_ge8": 1485, "single_feature_auc": {"amount": 0.833, "hour_night": 0.534, "receiver_age_days": 0.628, "sender_age_days": 0.609}} 

T13 {"SC-01_key_success": true, "SC-02_key_success": true, "SC-03_key_success": true, "SC-04_key_success": true, "SC-05_key_success": 

## Daily rhythm: salary days, Friday, Eid

In [6]:
x = tx.merge(lab[["txn_id", "is_fraud"]], on="txn_id")
day = x.groupby([x.ts.dt.floor("D"), "txn_type"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(9, 3.6))
for i, t in enumerate(["SEND_MONEY", "MOBILE_RECHARGE", "PAYMENT", "CASH_IN"]):
    ax.plot(day.index, day[t], color=plots.SERIES[i], label=t.replace("_", " ").title())
eid = [pd.Timestamp(cfg["world"]["start_date"]) + pd.Timedelta(days=d - 1) for d in cfg["world"]["eid_days"]]
ax.axvspan(eid[0], eid[-1] + pd.Timedelta(days=1), color=plots.GRID, alpha=0.8, linewidth=0)
ax.text(eid[0], ax.get_ylim()[1] * 0.95, " Eid", color=plots.INK2, fontsize=9, va="top")
ax.set_title("Transactions per day by type"); ax.set_ylabel("Transactions"); ax.legend(ncol=4, loc="lower right")
plt.tight_layout(); plt.show()

## Hour of day: fraud is NOT only a night problem

In [7]:
sc = x[x.txn_type.isin(["SEND_MONEY", "CASH_OUT", "PAYMENT", "ADD_MONEY"])]
h = pd.crosstab(sc.ts.dt.hour, sc.is_fraud, normalize="columns")
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(h.index, h[0], color=plots.SERIES[0], label="legit")
ax.plot(h.index, h[1], color=plots.SERIES[1], label="fraud")
ax.set_xlabel("Hour"); ax.set_ylabel("Share of transactions"); ax.set_title("Hourly profile, legit vs fraud")
ax.legend(); plt.tight_layout(); plt.show()

## Fraud by scenario and split (every scenario appears in train, val and test)

In [8]:
fr = lab[lab.is_fraud == 1]
pd.crosstab(fr.scenario, fr.split).reindex(columns=["train", "val", "test"])

split,train,val,test
scenario,,,
S1,577,193,174
S2,495,247,151
S3,211,105,66
S5,272,79,136
S6,187,51,70
S7,290,56,93
S8,121,46,44


In [9]:
data["cases"].groupby("scenario").agg(cases=("case_id", "size"), fraud_txns=("n_fraud_txns", "sum"),
                                       variants=("variant", lambda v: ", ".join(sorted(set(v) - {""}))))

,cases,fraud_txns,variants
scenario,,,
S1,191,943,"otp_call, register_harvest, remote_access_app"
S2,41,893,"job_fee, loan_fee, lottery, parcel_customs"
S3,61,383,
S5,41,487,"dress, eid_items, gadget, phone"
S6,56,308,
S7,166,439,"fake_upay_officer, jail_guard, lawyer, police,..."
S8,37,211,


## One planted case end to end: SC-01 SIM-swap takeover

In [10]:
ev = data["account_events"]
display(ev[ev.case_id == "SC-01"][["ts", "wallet_id", "event_type", "device_id", "area_id"]])
c = x.merge(lab[["txn_id", "case_id", "fraud_role"]], on="txn_id")
c[c.case_id == "SC-01"][["ts", "txn_type", "sender_id", "receiver_id", "amount", "status", "device_id", "fraud_role"]]

,ts,wallet_id,event_type,device_id,area_id
2833,2026-05-24 01:40:00,W0004539,SIM_SWAP,,CTG-01
2834,2026-05-24 01:52:00,W0004539,DEVICE_CHANGE,DV0013032,CTG-01
2835,2026-05-24 01:53:00,W0004539,PIN_RESET,DV0013032,CTG-01


,ts,txn_type,sender_id,receiver_id,amount,status,device_id,fraud_role
608619,2026-05-24 01:55:00,SEND_MONEY,W0004539,W0001994,48000.0,SUCCESS,DV0013032,takeover_drain
608630,2026-05-24 02:07:00,SEND_MONEY,W0001994,W0011288,47000.0,SUCCESS,DV0013037,mule_hop
608660,2026-05-24 02:30:00,CASH_OUT,W0011288,A00044,29000.0,SUCCESS,DV0013035,mule_cashout
608669,2026-05-24 02:41:00,SEND_MONEY,W0011288,W0011289,17000.0,SUCCESS,DV0013035,mule_hop
608703,2026-05-24 03:05:00,CASH_OUT,W0011289,A00044,16500.0,SUCCESS,DV0013036,mule_cashout


## Population (used for the fairness audit only, never as model inputs)

In [11]:
pd.concat({k: cust[k].value_counts(normalize=True).round(3) for k in ["segment", "kyc_level", "channel", "gender", "age_band", "urban_rural"]}).to_frame("share")

share
segment     salaried           0.235
            student            0.144
            small_business     0.140
            garment_worker     0.132
            homemaker          0.104
            remittance_family  0.097
            farmer_rural       0.089
            gig_rider          0.060
kyc_level   KYC2               0.776
            KYC1               0.224
channel     APP                0.833
            USSD               0.167
gender      M                  0.589
            F                  0.411
age_band    25-34              0.322
            18-24              0.239
            35-44              0.225
            45-59              0.160
            60+                0.054
urban_rural urban              0.507
            rural              0.493